# TiA 3 — Inductive bias and equivariance

**Big question:** If two models are expressive enough, why can one learn an image task with far fewer examples?

We isolate locality and weight sharing using synthetic bars; no external image dataset is needed.

## How to work through this activity

This is a guided investigation rather than a coding tutorial. For each experiment:

1. Read the mathematical claim and identify the quantity being measured.
2. Predict the qualitative result before running the code.
3. Run one cell at a time and inspect both values and plots.
4. Change only the suggested variable; rerun and explain what changed.
5. Answer the **Explain** questions in your own words.

The code contains more comments than production software intentionally. You are not expected to memorise framework syntax. Focus on the relationship between assumptions, measurements and conclusions.

## Notation and prediction

An inductive bias favours some functions before the data have uniquely determined one. For translation operator $T_\delta$, a feature map $f$ is translation-equivariant when

$$f(T_\delta x)=T_\delta f(x),$$

and a classifier $g$ is invariant when $g(T_\delta x)=g(x)$. Convolution shares a local kernel across positions; global pooling can turn equivariant features into an invariant decision. Predict which model will extrapolate from centred bars to displaced bars.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Fix every random-number generator so that your plots match the reference run.
# After completing the guided activity, change the seed to test robustness.
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

# The default path is designed for a CPU. Set this to False only after the
# notebook works and you want to run longer variants.
FAST_MODE = True

def make_shape_images(n=600, size=12, centred=False, seed=7):
    """Create noisy horizontal/vertical bars without downloading data."""
    local_rng = np.random.default_rng(seed)
    images = local_rng.normal(0, 0.12, (n, 1, size, size)).astype("float32")
    labels = local_rng.integers(0, 2, n)
    positions = np.full(n, size // 2) if centred else local_rng.integers(2, size - 2, n)
    for image, label, position in zip(images, labels, positions):
        image[0, position, 2:-2] += label == 0
        image[0, 2:-2, position] += label == 1
    return images, labels.astype("int64")

from torch import nn

In [ ]:
Xtr,ytr=make_shape_images(120,centred=True); Xte,yte=make_shape_images(500,centred=False,seed=8)
class MLP(nn.Module):
    def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Flatten(),nn.Linear(144,24),nn.ReLU(),nn.Linear(24,2))
    def forward(self,x): return self.net(x)
class CNN(nn.Module):
    def __init__(self): super().__init__(); self.features=nn.Sequential(nn.Conv2d(1,4,3,padding=1),nn.ReLU()); self.head=nn.Linear(4,2)
    def forward(self,x): return self.head(self.features(x).mean((2,3)))
def train(model,epochs=60):
    x=torch.tensor(Xtr); y=torch.tensor(ytr); opt=torch.optim.Adam(model.parameters(),lr=.03)
    for _ in range(epochs): opt.zero_grad(); loss=nn.functional.cross_entropy(model(x),y); loss.backward(); opt.step()
    with torch.no_grad(): return (model(torch.tensor(Xte)).argmax(1).numpy()==yte).mean()
results={}
for model in [MLP(),CNN()]:
    # Both see only centred training bars; testing moves bars to unseen positions.
    results[type(model).__name__]=train(model)
    print(type(model).__name__,sum(p.numel() for p in model.parameters()),f"shifted-position accuracy={results[type(model).__name__]:.3f}")
assert results["CNN"] > .90
assert results["CNN"] > results["MLP"]+.30

## Measure equivariance directly

For translation $T$ and representation $f$, compute $\|f(Tx)-Tf(x)\|/\|f(x)\|$. Cropping at boundaries is excluded from the comparison.

In [ ]:
cnn=CNN(); x=torch.tensor(Xte[:32]); shift=lambda z:torch.roll(z,2,dims=-1)
with torch.no_grad():
    fx=cnn.features(x); lhs=cnn.features(shift(x)); rhs=shift(fx)
    interior=(lhs[:,:,:,2:-2]-rhs[:,:,:,2:-2]).norm()/rhs[:,:,:,2:-2].norm()
print(f"CNN representation equivariance error: {interior:.2e}")
assert interior < .10

# Break the assumption: label is whether the bar lies left/right of centre.
Xp,_=make_shape_images(500,seed=12); pos=Xp[:,0].sum(1).argmax(1); yp=(pos>=6).astype("int64")
print("Global average pooling deliberately discards the absolute position needed by this task.")

### Explain

1. Distinguish invariance from equivariance.
2. Why does global pooling help the first task and hurt the absolute-position task?
3. State the conditions under which convolution improves sample efficiency.

**Reading:** [Bronstein et al., Geometric Deep Learning](https://arxiv.org/abs/2104.13478), Sections 2–3.

## Expected pattern and limits

The CNN should generalise from centred to displaced bars with far fewer parameters, and its interior feature-map equivariance error should be small. Global pooling then deliberately fails when absolute position defines the label. Convolution helps because its symmetry matches the first task, not because CNNs dominate every image problem.